### Task: Educatuin purpose. How Transformer should feed data?

    1. How to feed data correctly?
    1.1. What to do with a big time-series? (We already can split data by 512 sequence).
    1.2. How to apply time-positional encoding?


# Import libraries

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch.optim import Adam
from tqdm import tqdm
from IPython.display import clear_output

# Create Example Data

Let`s image that we are working with kind of 1D time series dataset.


1. **Input Preparation**:
   - Your 8 words, each with a 512-length embedding, are stacked into a tensor.
   - Shape: 8, batch_size, 512 (e.g., batchsize=1 for a single sequence).
   - Positional encodings are added (as above), keeping the shape.
   
2. **Attention Layer** (e.g., nn.MultiheadAttention):
   - Expects: [seqlen, batchsize, dmodel] (here, 8, batch_size, 512).
   - Query (Q), Key (K), Value (V) matrices are derived from this input via linear projections.
   - Self-attention computes relationships between all 8 positions (seqlen × seqlen attention matrix).

In [2]:
x_example = torch.rand(1, 16, 32) # batch, number_of_words, token feature size
print(x_example.shape)

# In other words we have 16 sequence where each of them has length 32.

torch.Size([1, 16, 32])


In [3]:
x_example[0]

tensor([[4.7277e-01, 9.0703e-02, 5.7039e-01, 6.7458e-01, 4.6467e-01, 7.7323e-01,
         9.1035e-01, 8.9322e-01, 5.8820e-01, 7.2337e-02, 9.0073e-01, 3.5170e-01,
         2.9882e-02, 4.0545e-01, 1.5930e-01, 5.7622e-01, 5.6649e-02, 1.6409e-01,
         4.2357e-01, 4.4091e-01, 5.3678e-01, 7.9362e-01, 4.3206e-01, 9.9590e-01,
         7.1491e-01, 9.1008e-02, 2.1952e-02, 4.6442e-01, 6.8629e-01, 9.7495e-01,
         6.1051e-01, 3.5873e-01],
        [2.6385e-01, 9.0447e-01, 7.2332e-01, 5.9668e-01, 9.3200e-02, 4.5144e-01,
         7.4649e-01, 7.2978e-01, 1.3273e-01, 2.1930e-01, 2.4590e-01, 3.3089e-02,
         3.7249e-01, 5.6521e-01, 5.0368e-01, 8.4708e-01, 8.7048e-01, 9.2512e-02,
         8.7248e-02, 8.9587e-01, 9.5825e-02, 1.4264e-01, 1.9360e-01, 7.8670e-01,
         7.0228e-01, 3.7899e-01, 4.8493e-01, 2.6625e-01, 1.1756e-01, 3.1738e-01,
         2.8333e-01, 6.0382e-01],
        [7.9486e-01, 1.4092e-01, 4.4297e-01, 2.0221e-02, 1.8871e-01, 2.3324e-03,
         3.1850e-01, 4.7712e-01, 7.7077e-

In [5]:
y_example = torch.rand(1, 16, 32) # batch, number_of_words, token feature size

We would like to apply Positional Encoding

https://discuss.pytorch.org/t/how-to-modify-the-positional-encoding-in-torch-nn-transformer/104308/2

The problem here for 1D time series that we have to input the whole time-series to get time embeddings.

In [10]:
class PositionalEncoding(nn.Module):

    def __init__(self, d_model, dropout=0.1, max_len=1024):
        """ 
        Notes:
            d_model: number of features which reprsenet 1 token in NLP.
            max_len: length of the longest sequence which we can hold in memory by POSINCODING!
        
        Example:
            max_len 1024 means that max pos len is 1024 for time series which has 16 sequence where 
            each sequence length is 32. Like torch.Size([1, 16, 32]) # batch, number sequences, feature sequence
        """
        super().__init__()
        self.dropout = nn.Dropout(p=dropout)

        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-np.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0).transpose(0, 1)
        self.register_buffer('pe', pe)

    def forward(self, x):
        x = x + self.pe[:x.size(0), :]
        return x

In [11]:
PositionalEncoder = PositionalEncoding(d_model=32)

In [12]:
x_pos_example = PositionalEncoder.forward(x_example)

In [13]:
x_pos_example.shape

torch.Size([1, 16, 32])

What should be used for DECODER layer?

Start point is NLP task! We need to decode sequence from one abstraction to another!

1. Hypothesis that we need to use prediction from the previous steps which will be indicate where CPs occured in the past.

2. Hypothesis that we need to use time series from the past steps && current step again??!

3. Hypothesis that we need to apply another CPs method to increase results?

No mask shoud be applied!

In [14]:
x_cps = torch.zeros_like(x_example) + 1e-4

In [15]:
x_cps[0]

tensor([[1.0000e-04, 1.0000e-04, 1.0000e-04, 1.0000e-04, 1.0000e-04, 1.0000e-04,
         1.0000e-04, 1.0000e-04, 1.0000e-04, 1.0000e-04, 1.0000e-04, 1.0000e-04,
         1.0000e-04, 1.0000e-04, 1.0000e-04, 1.0000e-04, 1.0000e-04, 1.0000e-04,
         1.0000e-04, 1.0000e-04, 1.0000e-04, 1.0000e-04, 1.0000e-04, 1.0000e-04,
         1.0000e-04, 1.0000e-04, 1.0000e-04, 1.0000e-04, 1.0000e-04, 1.0000e-04,
         1.0000e-04, 1.0000e-04],
        [1.0000e-04, 1.0000e-04, 1.0000e-04, 1.0000e-04, 1.0000e-04, 1.0000e-04,
         1.0000e-04, 1.0000e-04, 1.0000e-04, 1.0000e-04, 1.0000e-04, 1.0000e-04,
         1.0000e-04, 1.0000e-04, 1.0000e-04, 1.0000e-04, 1.0000e-04, 1.0000e-04,
         1.0000e-04, 1.0000e-04, 1.0000e-04, 1.0000e-04, 1.0000e-04, 1.0000e-04,
         1.0000e-04, 1.0000e-04, 1.0000e-04, 1.0000e-04, 1.0000e-04, 1.0000e-04,
         1.0000e-04, 1.0000e-04],
        [1.0000e-04, 1.0000e-04, 1.0000e-04, 1.0000e-04, 1.0000e-04, 1.0000e-04,
         1.0000e-04, 1.0000e-04, 1.0000e-

In [16]:
x_pos_cps = PositionalEncoder.forward(x_cps)

# Define Neural Network

https://pytorch.org/docs/stable/generated/torch.nn.TransformerDecoderLayer.html

https://pytorch.org/docs/stable/generated/torch.nn.TransformerEncoderLayer.html

In [18]:
class PositionalEncoding(nn.Module):

    def __init__(self, d_model, dropout=0.1, max_len=1024):
        """ 
        Notes:
            d_model: number of features which reprsenet 1 token in NLP.
            max_len: length of the longest sequence which we can hold in memory by POSINCODING!
        
        Example:
            max_len 1024 means that max pos len is 1024 for time series which has 16 sequence where 
            each sequence length is 32. Like torch.Size([16, 1, 32]) # number sequences, batch, feature sequence
        """
        super().__init__()
        self.dropout = nn.Dropout(p=dropout)

        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-np.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0).transpose(0, 1)
        self.register_buffer('pe', pe)

    def forward(self, x):
        x = x + self.pe[:x.size(0), :]
        return x

class TransformerCPD(nn.Module):
    def __init__(self,
                 sequence_length: int = 32,
                 n_layers: int = 4,
                 n_heads: int = 2,
                 dim_feedforward: int = 1024):
        super().__init__()
        
        self.activation = nn.Sigmoid()
        
        encoder_layer = nn.TransformerEncoderLayer(d_model=sequence_length,
                                                   nhead=n_heads,
                                                   dim_feedforward=dim_feedforward,
                                                   norm_first=True,
                                                   batch_first=True)
        decoder_layer = nn.TransformerDecoderLayer(d_model=sequence_length,
                                                   nhead=n_heads,
                                                   dim_feedforward=dim_feedforward,
                                                   norm_first=True,
                                                   batch_first=True)
        
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=n_layers)
        self.decoder = nn.TransformerDecoder(decoder_layer, num_layers=n_layers)

    @torch.inference_mode()
    def encode(self, x):
        """ Encode time series to a map of features.

        Arg:
            x: batch of raw time series data.
            
        Return:
            batch of encoded time series data.
        """
        return self.encoder(x)

    def forward(self, encoder_x, encoder_pad_mask, decoder_x, decoder_pad_mask):
        encoded_data = self.encoder(src=encoder_x,
                                    src_key_padding_mask=encoder_pad_mask)
        decoded_data = self.decoder(memory=encoded_data,
                                    memory_key_padding_mask=encoder_pad_mask,
                                    tgt=decoder_x,
                                    tgt_key_padding_mask=decoder_pad_mask
                                   )
        return self.activation(decoded_data)

def mask_padding(x: torch.Tensor, mask_val: int = -9999) -> torch.Tensor:
    # Reduce to 2D: Check any dim for padding value, collapse to (batch_size, seq_len)
    return (x == mask_val).any(dim=-1)  # True where token is padded

# Example Inference

In [19]:
model = TransformerCPD()

/home/gishb/PycharmProjects/DirectionalDrillingChangePointDetection/.venv/lib/python3.10/site-packages/torch/nn/modules/transformer.py:385: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  warnings.warn(


In [21]:
encoder_x_mask_pad = mask_padding(x_example)
decoder_x_mask_pad = mask_padding(x_cps)

out = model(encoder_x=x_pos_example, encoder_pad_mask=encoder_x_mask_pad,
      decoder_x=x_pos_cps, decoder_pad_mask=decoder_x_mask_pad)
print(out.shape)

torch.Size([1, 16, 32])


In [22]:
out

tensor([[[9.9607e-01, 2.0269e-01, 9.9311e-01, 4.6603e-01, 4.4301e-01,
          7.8877e-01, 5.0027e-01, 1.6604e-01, 5.1647e-01, 7.5432e-01,
          1.6632e-02, 9.0410e-01, 1.5984e-01, 9.7790e-01, 8.2548e-01,
          8.6408e-01, 9.0767e-01, 3.6255e-01, 1.0740e-01, 7.5895e-01,
          4.1640e-01, 9.2879e-01, 6.1262e-01, 9.8831e-01, 2.9490e-04,
          9.9428e-01, 9.9247e-01, 1.1761e-01, 7.0271e-01, 8.9246e-01,
          9.8897e-01, 9.9779e-01],
         [9.9641e-01, 1.8393e-01, 9.9430e-01, 8.4203e-01, 4.8233e-01,
          8.1622e-01, 7.6598e-01, 1.8679e-01, 5.0884e-01, 6.4650e-01,
          1.4960e-02, 9.3378e-01, 5.0730e-02, 9.6924e-01, 6.9125e-01,
          7.6369e-01, 9.3384e-01, 7.5520e-01, 1.1332e-01, 7.7747e-01,
          4.5554e-01, 9.4902e-01, 4.9334e-01, 9.9473e-01, 9.3252e-04,
          9.5962e-01, 9.9608e-01, 8.0172e-02, 7.4432e-01, 8.9974e-01,
          9.8076e-01, 9.9605e-01],
         [9.8042e-01, 6.9160e-02, 9.8168e-01, 4.7236e-01, 4.3467e-01,
          6.7030e-01

When should we use mask?

Mask used to hide useless params when we used padding!

https://sanjayasubedi.com.np/deeplearning/masking-in-attention/

### Why This Works
- `x == mask_val`: 3D boolean tensor (batch_size, seq_len, embed_dim).
- `.any(dim=-1)`: Collapses the embed_dim axis. If any value in a token’s embedding matches `mask_val`, it’s padded → 2D (batch_size, seq_len).
- Matches `torch.nn.MultiheadAttention` expectation.

In [47]:
# idea to use mask for input sequence to hide future padding
test = torch.ones(1, 4)

In [45]:
test[0][-1] = -9999

In [46]:
test

tensor([[ 1.0000e+00,  1.0000e+00,  1.0000e+00, -9.9990e+03]])

In [35]:
def mask_padding(x: torch.Tensor, mask_val: int = -9999) -> torch.Tensor:
    # Reduce to 2D: Check any dim for padding value, collapse to (batch_size, seq_len)
    return (x == mask_val).any(dim=-1)  # True where token is padded

In [44]:
mask_padding(test)

tensor([[False, False, False,  True]])